# HeatAhead: assigned weather-band analysis

**Start here.** Put this notebook beside `days.csv` and `buildings.csv`, or keep the CSVs in your repository's
`HeatAhead_weather_2017_onward/` folder. Open in Python 3 and run one cell at a time with **Shift+Enter**.
Read each result before moving on. Required packages: pandas, numpy, matplotlib.

This notebook completes the weather-band analysis and audits the input needed for the three building-specific tasks.
It deliberately leaves building-by-weather rates uncomputed because the supplied property file has no complaint dates.
Daily aggregate counts cannot tell us which properties complained on particular dates.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
 from IPython.display import display
except ImportError:
 display=print

def find_input(name):
 roots=[Path.cwd(),*list(Path.cwd().parents)[:2]]
 for root in roots:
  for subdir in ['', 'HeatAhead_weather_2017_onward','data','upload']:
   p=root/subdir/name
   if p.is_file(): return p
 raise FileNotFoundError('Upload '+name+' beside this notebook, or put it in HeatAhead_weather_2017_onward/.')

OUT=Path('analysis_outputs')
OUT.mkdir(exist_ok=True)
days=pd.read_csv(find_input('days.csv'),parse_dates=['day']).sort_values('day')
buildings=pd.read_csv(find_input('buildings.csv'),dtype={'bbl':'string','zipcode':'string'})
print('Daily rows:',len(days),'| dates:',days.day.min().date(),'to',days.day.max().date())
print('Property/lot rows:',len(buildings))
print('Property columns:',list(buildings.columns))


## 1. Apply the assigned weather bands

Default: Frigid <20, Freezing 20 to <32, Cold 32 to <45, Mild >=45 on TMIN.
These are the assignment's Fahrenheit thresholds; confirm that the weather source was exported in Fahrenheit.
The supplied TMIN observations are integers. If you widen Frigid to <25, Freezing must start at 25 to avoid overlap.

Begin with the original cutoff. There are 88 Frigid observations, so first check group-level sparsity before widening.
Use heating-season rows for the primary cold-versus-mild comparison. Also retain the full-period counts for context.


In [ ]:
FRIGID_CUTOFF=20  # Change to 25 only if your team agrees; Freezing then becomes 25 to <32.
if FRIGID_CUTOFF not in [20,25]:
 raise ValueError('Use the assigned cutoff 20 or alternative 25.')
if days['day'].duplicated().any():
 raise ValueError('Duplicate daily dates: check the input.')
days['TMIN']=pd.to_numeric(days['TMIN'],errors='coerce')
days['complaints']=pd.to_numeric(days['complaints'],errors='coerce')
if days[['TMIN','complaints']].isna().any().any():
 raise ValueError('Missing temperatures or complaint counts need review before band summaries.')
labels=['Frigid','Freezing','Cold','Mild']
days['weather_band']=pd.cut(days['TMIN'],bins=[-np.inf,FRIGID_CUTOFF,32,45,np.inf],right=False,labels=labels)
# Parse the supplied heat_season flag explicitly so strings such as FALSE cannot become True accidentally.
heat_flag=days['heat_season'].astype(str).str.lower().map({'true':True,'false':False,'1':True,'0':False})
if heat_flag.isna().any():
 raise ValueError('Unexpected heat_season flag. Inspect its values.')
days['analysis_heat_season']=heat_flag

def band_summary(frame):
 t=frame.groupby('weather_band',observed=False).agg(
  days=('day','size'),complaint_count=('complaints','sum'),
  mean_complaints_per_day=('complaints','mean'),median_complaints_per_day=('complaints','median'))
 mild=t.loc['Mild','mean_complaints_per_day']
 t['daily_volume_vs_mild']=t['mean_complaints_per_day']/mild if mild>0 else np.nan
 return t

all_days=band_summary(days)
heating_days=band_summary(days.loc[days['analysis_heat_season']])
print('ALL AVAILABLE DAYS')
display(all_days.round(2))
print('HEATING-SEASON ROWS ONLY: recommended primary comparison')
display(heating_days.round(2))
days[['day','TMIN','weather_band','analysis_heat_season','complaints']].to_csv(OUT/'days_with_weather_bands.csv',index=False)
all_days.to_csv(OUT/'weather_bands_all_days.csv')
heating_days.to_csv(OUT/'weather_bands_heating_season.csv')


**Interpretation:** these means are aggregate complaints per day. They are not a property's probability of complaining.
The aggregate `daily_volume_vs_mild` column is a descriptive volume ratio, not the requested building-specific multiplier.
Summer mild days have a different complaint baseline. Keeping a common season makes the comparison more meaningful,
although season, year, reporting patterns, and other factors still differ.


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4.8))
all_days['days'].plot.bar(ax=axes[0],rot=0,color='#28648b')
heating_days['mean_complaints_per_day'].plot.bar(ax=axes[1],rot=0,color='#d4772c')
axes[0].set(title='Number of observed days in each band',xlabel='',ylabel='Days, full daily file')
axes[1].set(title='Complaint volume by band: heating-season rows',xlabel='',ylabel='Mean aggregate complaints per day')
for ax in axes:
 ax.spines['top'].set_visible(False)
 ax.spines['right'].set_visible(False)
 ax.grid(axis='y',alpha=.15)
fig.tight_layout()
fig.savefig(OUT/'weather_band_summary.png',dpi=160,bbox_inches='tight')
plt.show()

year_band=days.groupby([days['day'].dt.year.rename('year'),'weather_band'],observed=False).size().unstack(fill_value=0)
print('Days per band by year: partial first and last years are included.')
display(year_band)
year_band.to_csv(OUT/'weather_band_days_by_year.csv')


## 2. Audit the missing building-by-date connection

The requested heatmap, sensitivity multipliers, and individual property histories need a third table.
Ask the data preparer for **one row per property per complaint date**, or raw complaint events that can be grouped that way.

Minimum clean columns:
- `bbl`: property key matching buildings.csv, read as text.
- `complaint_day`: the actual complaint date, preferably YYYY-MM-DD in New York local time.
- `complaint_count`: qualifying complaints on that property/date; optional if each row is one event.

Confirm complaint category filters, date coverage, missing-source dates, and identifier mappings. If the raw data
only have HPD BuildingID, request the BuildingID-to-BBL mapping. An aggregated building-day table is enough;
you do not need to transfer every individual report if the preparer can supply correct counts.


In [ ]:
missing_dates=pd.date_range(days.day.min(),days.day.max()).difference(days.day)
audit={
 'daily_start':days.day.min().date().isoformat(),'daily_end':days.day.max().date().isoformat(),
 'absent_daily_dates':[d.date().isoformat() for d in missing_dates],
 'property_keys_unique':not buildings.bbl.duplicated().any(),
 'property_rows':len(buildings),
 'daily_total_complaints':int(days.complaints.sum()),
 'property_total_complaints':int(buildings.complaints.sum()),
 'missing_input':'Dated complaint events or building-day counts linked to BBL',
 'building_by_band_results':'Not computed: date-linked property outcomes are absent'
}
print(json.dumps(audit,indent=2))
(OUT/'analysis_data_audit.json').write_text(json.dumps(audit,indent=2),encoding='utf-8')
requirements="""HEATAHEAD BUILDING-BAND DATA REQUEST
Provide a CSV with bbl, complaint_day, and complaint_count.
One row per BBL/date is sufficient. Raw events are also acceptable.
Use the qualifying complaint categories agreed by the team and document their filters.
Confirm date coverage, date timezone, missing-source dates, and any BuildingID-to-BBL mapping.
Provide property coverage/active dates if available; denominator eligibility must be documented.
Current buildings.csv contains only totals, so property-by-weather results cannot yet be computed.
"""
(OUT/'building_day_data_request.txt').write_text(requirements,encoding='utf-8')


## 3. Agree on the rate before making the heatmap

For the primary analysis, define a **complaint-day rate**:

**Property rate in a band = days with at least one qualifying complaint / eligible observed days in that band.**

For a size/era/alteration group, pool the numerator over its properties and divide by eligible property-days.
Count each property at most once per day for this rate. Keep raw complaint volume as a separate metric.
Zero-event days must be included in exposure; a complaints-only dataset cannot supply the denominator by itself.

Agree on the window and eligibility first. A current property snapshot does not establish that every property existed,
was covered by the source, or was active throughout 2017 onward. Use known coverage/active dates when available.
A restricted cohort known to predate the analysis window is a possible documented exploratory assumption.
Use the same qualifying categories and common coverage dates in the weather and event data.


## 4. What the three pending deliverables will show

| Output | Calculation | Interpretation |
|---|---|---|
| Type-by-weather heatmap | Complaint-days / eligible property-days for each size, era, and alteration group | Which groups report problems more often in each weather band |
| Cold-sensitivity table | Frigid complaint-day rate / Mild complaint-day rate, with both rates and counts shown | Whether observed complaints are concentrated on cold days or also occur on mild days |
| Individual property history | Complaint-day rate and numerator/denominator for each BBL and weather band | For example, 8 complaint-days among 40 eligible frigid days = 20%, not 8 out of all complaints |

Use three separate heatmaps (size, era, altered) first; crossing all three can create sparse groups.
Era boundaries should be agreed by the team. `yearalter1=0` means no recorded alteration in this extract,
not proof that the property was never altered. A historical analysis must avoid using an alteration that happened
after the complaint date as if it were already known.

If the mild-day numerator is zero, show the raw rates/counts and mark the raw ratio unavailable.
Do not invent infinity as a useful sensitivity score. Low counts require uncertainty or a clearly labeled smoothing method.
The chronic-versus-cold-triggered labels should use both absolute rates and the ratio; a large ratio alone can reflect
very few events. These categories are exploratory and need explicit rules rather than arbitrary claims of validation.


## 5. Hand off what is ready now

Send the weather-band chart, the two weather summary tables, and the data request to your teammates.
The original cutoff gives 88 Frigid days overall. This is enough to start exploration, while individual property
groups or shorter training periods may still be sparse.

After the dated file arrives, join complaint_day to day for weather, then bbl to property attributes.
Do not join only on a borough or apply the aggregate weather pattern to every property as if it were observed history.
For predictive use, calculate each property's history strictly before the prediction date.


In [ ]:
print('Ready to send from analysis_outputs/:')
for name in ['weather_band_summary.png','weather_bands_all_days.csv','weather_bands_heating_season.csv',
             'weather_band_days_by_year.csv','building_day_data_request.txt']:
 print((OUT/name).as_posix())
print('Next input needed: dated property complaint events or building-day counts linked to BBL.')
